# Study 904 — Shareholder-Yield + Quality — the teardown

The excess-of-cash Sharpe race, the QSY-minus-RAW and QSY-minus-SPY HAC *t*, the paired block-bootstrap Sharpe-gap CIs, the era cut, the costed sleeves, and the 20-seed synthetic control.

In [1]:
R = {'start': '2013-08', 'end': '2026-06', 'n_months': 155, 'fingerprint': '6ab3e873ef9c', 'qsy_cagr': 12.96, 'qsy_vol': 15.4, 'qsy_sharpe': 0.764, 'qsy_maxdd': -25.1, 'qsy_wealth': 4.83, 'raw_cagr': 12.02, 'raw_vol': 17.0, 'raw_sharpe': 0.657, 'raw_maxdd': -29.3, 'raw_wealth': 4.33, 'spy_cagr': 14.14, 'spy_vol': 14.5, 'spy_sharpe': 0.874, 'spy_maxdd': -23.9, 'spy_wealth': 5.52, 'qr_gap': 0.106, 'qr_diff_bps': 4.94, 'qr_diff_ann': 0.59, 'qr_t1s': 0.55, 'qr_tnw': 0.57, 'qr_ci_lo': -0.013, 'qr_ci_hi': 0.241, 'qr_pneg': 0.04, 'qs_gap': -0.111, 'qs_diff_ann': -0.91, 'qs_diff_bps': -7.62, 'qs_t1s': -0.79, 'qs_tnw': -0.9, 'qs_ci_lo': -0.243, 'qs_ci_hi': 0.013, 'qs_pneg': 0.96, 'rs_gap': -0.217, 'rs_diff_ann': -1.51, 'rs_tnw': -0.79, 'qr_e_gap': 0.163, 'qr_e_diff': 0.92, 'qr_e_t': 1.12, 'qr_e_n': 77, 'qr_l_gap': 0.073, 'qr_l_diff': 0.27, 'qr_l_t': 0.14, 'qr_l_n': 78, 'qs_e_gap': -0.112, 'qs_e_diff': -0.45, 'qs_e_t': -0.55, 'qs_e_n': 77, 'qs_l_gap': -0.117, 'qs_l_diff': -1.37, 'qs_l_t': -0.72, 'qs_l_n': 78, 'qsy_turn': 0.76, 'qsy_drag': 0.3, 'qsy_net': 0.763, 'raw_turn': 0.32, 'raw_drag': 0.1, 'raw_net': 0.657, 'spyd_n': 128, 'spyd_qsy_sh': 0.716, 'spyd_sh': 0.476, 'spyd_gap': 0.24, 'spyd_t': 0.86, 'null_fire': 1, 'null_seed_t': -0.85, 'planted_gap': 20.46, 'planted_t': 168.4, 'cal_years': [2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025], 'cal_qsy': [-8.1, 34.0, 12.8, 29.8, -15.4, 24.1, 20.0, 15.3], 'cal_raw': [-10.5, 34.1, 8.4, 32.6, -10.2, 17.2, 17.3, 17.9], 'cal_spy': [-4.6, 31.2, 18.3, 28.7, -18.2, 26.2, 24.9, 17.7]}

## The race — common window, excess of cash (minus BIL)

In [2]:
for name in ('qsy','raw','spy'):
    lbl = {'qsy':'QSY (PKW+QUAL)','raw':'RAW (PKW)     ','spy':'SPY           '}[name]
    print(f"{lbl}: CAGR {R[name+'_cagr']:5.2f}%  vol {R[name+'_vol']:4.1f}%  "
          f"exSharpe {R[name+'_sharpe']:.3f}  maxDD {R[name+'_maxdd']:6.1f}%  "
          f"$1->{R[name+'_wealth']:.2f}")

QSY (PKW+QUAL): CAGR 12.96%  vol 15.4%  exSharpe 0.764  maxDD  -25.1%  $1->4.83
RAW (PKW)     : CAGR 12.02%  vol 17.0%  exSharpe 0.657  maxDD  -29.3%  $1->4.33
SPY           : CAGR 14.14%  vol 14.5%  exSharpe 0.874  maxDD  -23.9%  $1->5.52


## Race 2 — does the quality overlay add value over raw buybacks? (QSY − RAW)

The QSY−RAW monthly spread is cash-independent (cash cancels), so its mean / HAC *t* is the clean 'does the overlay out-earn raw buybacks?' statistic.

In [3]:
print(f"excess Sharpe : QSY {R['qsy_sharpe']:.3f}  vs RAW {R['raw_sharpe']:.3f}  "
      f"-> GAP {R['qr_gap']:+.3f}")
print(f"QSY-minus-RAW : {R['qr_diff_bps']:+.2f} bps/mo ({R['qr_diff_ann']:+.2f}%/yr)  "
      f"one-sample t {R['qr_t1s']:+.2f}  NW t {R['qr_tnw']:+.2f}")
print(f"bootstrap gap : {R['qr_gap']:+.3f}  95% CI [{R['qr_ci_lo']:+.3f}, {R['qr_ci_hi']:+.3f}]  "
      f"P(gap<0) = {R['qr_pneg']:.2f}")

excess Sharpe : QSY 0.764  vs RAW 0.657  -> GAP +0.106
QSY-minus-RAW : +4.94 bps/mo (+0.59%/yr)  one-sample t +0.55  NW t +0.57
bootstrap gap : +0.106  95% CI [-0.013, +0.241]  P(gap<0) = 0.04


The overlay's advantage is **positive and the bootstrap is 96% positive**, but the HAC *t* is only **+0.57** — a real direction, not a certified premium.

## Race 1 — does quality-screened shareholder yield beat the market? (QSY − SPY)

In [4]:
print(f"QSY vs SPY: gap {R['qs_gap']:+.3f}  diff {R['qs_diff_ann']:+.2f}%/yr  "
      f"NW t {R['qs_tnw']:+.2f}  95% CI [{R['qs_ci_lo']:+.3f}, {R['qs_ci_hi']:+.3f}]  P(gap<0)={R['qs_pneg']:.2f}")
print(f"RAW vs SPY: gap {R['rs_gap']:+.3f}  diff {R['rs_diff_ann']:+.2f}%/yr  NW t {R['rs_tnw']:+.2f}")

QSY vs SPY: gap -0.111  diff -0.91%/yr  NW t -0.90  95% CI [-0.243, +0.013]  P(gap<0)=0.96
RAW vs SPY: gap -0.217  diff -1.51%/yr  NW t -0.79


**Both buyback sleeves trailed SPY** — the market-beat claim is the wrong sign (insignificantly). Owning plain SPY beat owning either buyback wrapper.

## Era cut (split 2020-01-01) — QSY vs RAW is positive in both halves, never significant

In [5]:
print('QSY - RAW:')
print(f"  2013-08..2019-12 (n={R['qr_e_n']}): gap {R['qr_e_gap']:+.3f}  diff {R['qr_e_diff']:+.2f}%/yr  NW t {R['qr_e_t']:+.2f}")
print(f"  2020-01..2026-06 (n={R['qr_l_n']}): gap {R['qr_l_gap']:+.3f}  diff {R['qr_l_diff']:+.2f}%/yr  NW t {R['qr_l_t']:+.2f}")
print('QSY - SPY:')
print(f"  2013-08..2019-12 (n={R['qs_e_n']}): gap {R['qs_e_gap']:+.3f}  diff {R['qs_e_diff']:+.2f}%/yr  NW t {R['qs_e_t']:+.2f}")
print(f"  2020-01..2026-06 (n={R['qs_l_n']}): gap {R['qs_l_gap']:+.3f}  diff {R['qs_l_diff']:+.2f}%/yr  NW t {R['qs_l_t']:+.2f}")

QSY - RAW:
  2013-08..2019-12 (n=77): gap +0.163  diff +0.92%/yr  NW t +1.12
  2020-01..2026-06 (n=78): gap +0.073  diff +0.27%/yr  NW t +0.14
QSY - SPY:
  2013-08..2019-12 (n=77): gap -0.112  diff -0.45%/yr  NW t -0.55
  2020-01..2026-06 (n=78): gap -0.117  diff -1.37%/yr  NW t -0.72


## Costed — monthly rebalance turnover x one-way spread (long-only, no borrow)

Turnover is only the drift of PKW/QUAL back to 50/50 (~0.8%/mo); raw buyback is a single ETF (no rebalance). Costs are a rounding error — not the story.

In [6]:
print(f"QSY: turnover {R['qsy_turn']:.2f}%/mo  drag {R['qsy_drag']:.1f} bps/yr  "
      f"gross exSharpe {R['qsy_sharpe']:.3f} -> net {R['qsy_net']:.3f}")
print(f"RAW: turnover {R['raw_turn']:.2f}%/mo  drag {R['raw_drag']:.1f} bps/yr  "
      f"gross exSharpe {R['raw_sharpe']:.3f} -> net {R['raw_net']:.3f}")

QSY: turnover 0.76%/mo  drag 0.3 bps/yr  gross exSharpe 0.764 -> net 0.763
RAW: turnover 0.32%/mo  drag 0.1 bps/yr  gross exSharpe 0.657 -> net 0.657


## Context — raw dividend yield (SPYD) and the too-young BUYB

In [7]:
print(f"QSY vs SPYD (raw dividend yield, from 2015-11, n={R['spyd_n']}): "
      f"exSharpe {R['spyd_qsy_sh']:.3f} vs {R['spyd_sh']:.3f}  gap {R['spyd_gap']:+.3f}  NW t {R['spyd_t']:+.2f}")
print('BUYB (standalone buyback ETF) lists 2026-05 -> 66 days: too young to race; named only')

QSY vs SPYD (raw dividend yield, from 2015-11, n=128): exSharpe 0.716 vs 0.476  gap +0.240  NW t +0.86
BUYB (standalone buyback ETF) lists 2026-05 -> 66 days: too young to race; named only


## Synthetic positive control — the machinery is unbiased

Live: the gap detector must NOT fire on the null and must recover a planted quality-over-raw premium.

In [8]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
import numpy as np
from sy_quality import data, strategy as st
null_t = np.array([st.synthetic_detect(data.synthetic_world(n_months=150, edge=0.0, seed=904+s))['t_nw'] for s in range(8)])
print(f"null (edge=0), 8 seeds: NW t mean {null_t.mean():+.2f} (sd {null_t.std(ddof=1):.2f}), |t|>=2 in {(abs(null_t)>=2).sum()}/8")
planted = st.synthetic_detect(data.synthetic_world(n_months=150, edge=3.0, seed=904))
print(f"planted (edge=+3%/yr): gap {planted['sharpe_gap']:+.3f}  NW t {planted['t_nw']:+.2f}  diff {planted['diff_ann_pct']:+.2f}%/yr")

null (edge=0), 8 seeds: NW t mean -0.54 (sd 0.83), |t|>=2 in 0/8
planted (edge=+3%/yr): gap +20.462  NW t +168.44  diff +298.50%/yr


## Verdict

- **Signal — Weak.** The quality overlay genuinely improves raw buybacks: a **-25.1%** vs **-29.3%** shallower crash and a Sharpe gap of **+0.106**, positive in *both* eras (gaps +0.163 / +0.073) with the bootstrap 96% positive (CI **[-0.013, +0.241]**, P(gap<0)=0.04). But it never clears the HAC bar (NW *t* = **+0.57**, no era significant), and — decisively — **neither buyback sleeve beats plain SPY** (QSY -0.91 pp/yr, *t* = -0.90; RAW -1.51 pp/yr). The synthetic control recovers a *planted* edge cleanly (*t* = 168, fires on 1/20 nulls), so a real premium *would* have shown — the market-beat one didn't. Short single-regime tape.
- **Tradability — Fragile.** The overlay is trivially buyable (cheap, liquid, long-only, rebalance drag 0.3 bps/yr — nothing erases it, not a Mirage) — but there is no significant premium to bank: you buy a shallower-drawdown *cleanup* of raw buybacks, not a certified edge, and the whole complex trails the market. Fragile is the honest stamp.